In [16]:
import json
import os
from datetime import datetime
import networkx as nx
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATASET_DIR = "dataset"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

with open(os.path.join(RESULTS_DIR, "cluster_summary.json"), "r", encoding="utf-8") as f:
    cluster_summary_data = json.load(f)

with open(os.path.join(DATASET_DIR, "services.json"), "r", encoding="utf-8") as f:
    topology_data = json.load(f)

with open(os.path.join(DATASET_DIR, "incidents_history.json"), "r", encoding="utf-8") as f:
    history_data = json.load(f)

alerts_detail = []
with open(os.path.join(DATASET_DIR, "alerts_sample.jsonl"), "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            alerts_detail.append(json.loads(line))

In [17]:
G = nx.DiGraph()

for svc in topology_data["services"]:
    G.add_node(svc["name"], type="service")

for store in topology_data["stores"]:
    G.add_node(store["name"], type="store")

for edge in topology_data["edges"]:
    G.add_edge(edge["from"], edge["to"])

In [18]:
def calculate_graph_temporal_rca(cluster_services, alerts_log, graph):
    subgraph = graph.subgraph(cluster_services).copy()

    graph_scores = {}
    for node in cluster_services:
        try:
            graph_scores[node] = 1.0 if subgraph.out_degree(node) == 0 else 0.2
        except:
            graph_scores[node] = 0.5

    earliest_ts = {}
    for alert in alerts_log:
        svc = alert["service"]
        if svc in cluster_services:
            ts_epoch = datetime.strptime(alert["ts"], "%Y-%m-%dT%H:%M:%SZ").timestamp()
            if svc not in earliest_ts or ts_epoch < earliest_ts[svc]:
                earliest_ts[svc] = ts_epoch
                
    if len(earliest_ts) > 1:
        min_ts = min(earliest_ts.values())
        max_ts = max(earliest_ts.values())
        ts_range = max_ts - min_ts if max_ts != min_ts else 1.0
        time_scores = {svc: (max_ts - ts) / ts_range for svc, ts in earliest_ts.items()}
    else:
        time_scores = {svc: 1.0 for svc in cluster_services}
        
    combined_candidates = []
    for svc in cluster_services:
        g_score = graph_scores.get(svc, 0.5)
        t_score = time_scores.get(svc, 0.0)
        final_score = 0.4 * g_score + 0.6 * t_score
        combined_candidates.append([svc, round(final_score, 2)])

    combined_candidates.sort(key=lambda x: x[1], reverse=True)
    return combined_candidates[:3]

In [19]:
import re

def clean_technical_text(text_input):
    tokens = re.split(r'[^a-zA-Z0-9_\-]+', text_input.lower())

    technical_stop_words = {
        'ms', 'p99', 'warn', 'crit', 'severity', 'metric', 'value', 'threshold',
        'labels', 'env', 'prod', 'region', 'ap-southeast-1', 'ts', 'id', 'z', 't',
        'latency', 'utilization', 'rate', 'time', 'ratio', 'unrelated', 'noise', 'independent'
    }

    cleaned_tokens = [t for t in tokens if t not in technical_stop_words and len(t) > 1]
    
    return " ".join(cleaned_tokens)

In [21]:
def retrieve_by_keyword_similarity_advanced(cluster_services, fingerprints, history_incidents, top_k=3):
    retrieved_incidents = []

    raw_current_text = " ".join(cluster_services) + " " + " ".join(fingerprints)
    set_current = set(clean_technical_text(raw_current_text).split())
    
    for inc in history_incidents:
        raw_history_text = " ".join(inc["services_involved"]) + " " + inc["summary"]
        set_history = set(clean_technical_text(raw_history_text).split())

        intersection = set_current.intersection(set_history)
        union = set_current.union(set_history)
        jaccard_score = len(intersection) / len(union) if union else 0.0
        
        retrieved_incidents.append({
            "id": inc["id"],
            "score": round(jaccard_score, 2),
            "class": inc["root_cause_class"],
            "actions": [inc["remediation"]]
        })
        
    retrieved_incidents.sort(key=lambda x: x["score"], reverse=True)
    return retrieved_incidents[:top_k]

In [22]:
final_output_payload = []

for cluster in cluster_summary_data["clusters"]:
    c_id = cluster["cluster_id"]
    c_services = cluster["services"]
    c_fingerprints = cluster["fingerprints"]

    graph_top3 = calculate_graph_temporal_rca(c_services, alerts_detail, G)
    if not graph_top3:
        continue
    predicted_root = graph_top3[0][0]

    matched_history = retrieve_by_keyword_similarity_advanced(c_services, c_fingerprints, history_data["incidents"], top_k=3)
    
    if matched_history and matched_history[0]["score"] > 0.0:
        best_match = matched_history[0]
        rc_class = best_match["class"]
        actions = best_match["actions"]
        confidence = best_match["score"]
        similar_ids = [case["id"] for case in matched_history if case["score"] > 0.0]
        method_tag = "graph+advanced_knn"
        reason_str = f"Mẫu đặc trưng lỗi trùng khớp chuẩn xác ngữ nghĩa với sự cố lịch sử {best_match['id']} sau khi áp dụng bộ lọc nhiễu hạ tầng."
    else:
        rc_class = "other"
        actions = ["Investigate manually"]
        confidence = graph_top3[0][1]
        similar_ids = []
        method_tag = "graph-only-fallback"

    final_output_payload.append({
        "cluster_id": c_id,
        "graph_top3": graph_top3,
        "root_cause": predicted_root,
        "class": rc_class,
        "confidence": confidence,
        "actions": actions,
        "reasoning": reason_str,
        "similar_incidents": similar_ids,
        "method": method_tag
    })

with open(os.path.join(RESULTS_DIR, "rca_output.json"), "w", encoding="utf-8") as f:
    json.dump({"clusters_analyzed": len(final_output_payload), "results": final_output_payload}, f, indent=2, ensure_ascii=False)

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

history_incidents = history_data["incidents"]
corpus_cleaned = []

for inc in history_incidents:
    raw_text = " ".join(inc["services_involved"]) + " " + inc["summary"]
    corpus_cleaned.append(clean_technical_text(raw_text))

vectorizer = TfidfVectorizer()
X_history_cleaned = vectorizer.fit_transform(corpus_cleaned)

def retrieve_by_tfidf_similarity_advanced(cluster_services, fingerprints, top_k=3):
    raw_current_text = " ".join(cluster_services) + " " + " ".join(fingerprints)
    cleaned_current_text = clean_technical_text(raw_current_text)
    
    X_current = vectorizer.transform([cleaned_current_text])
    
    cos_scores = cosine_similarity(X_current, X_history_cleaned).flatten()
    top_indices = cos_scores.argsort()[::-1][:top_k]
    
    bonus_retrieved = []
    for idx in top_indices:
        bonus_retrieved.append({
            "id": history_incidents[idx]["id"],
            "score": round(float(cos_scores[idx]), 2),
            "class": history_incidents[idx]["root_cause_class"],
            "actions": [history_incidents[idx]["remediation"]]
        })
    return bonus_retrieved

In [25]:
bonus_results = []

for cluster in cluster_summary_data["clusters"]:
    c_id = cluster["cluster_id"]
    c_services = cluster["services"]
    c_fingerprints = cluster["fingerprints"]
    
    graph_top3 = calculate_graph_temporal_rca(c_services, alerts_detail, G)
    predicted_root = graph_top3[0][0]

    tfidf_matches = retrieve_by_tfidf_similarity_advanced(c_services, c_fingerprints, top_k=3)
    
    if tfidf_matches and tfidf_matches[0]["score"] > 0.0:
        best_match = tfidf_matches[0]
        rc_class = best_match["class"]
        actions = best_match["actions"]
        confidence = best_match["score"]
        similar_ids = [case["id"] for case in tfidf_matches if case["score"] > 0.0]
        method_tag = "graph+tfidf_knn"
    else:
        rc_class = "other"
        actions = ["Investigate manually"]
        confidence = graph_top3[0][1]
        similar_ids = []
        method_tag = "graph-only-fallback"

    print(f"Cluster: {c_id} -> Chọn bằng TF-IDF: {rc_class} (Confidence: {confidence}) | Similar: {similar_ids}")

Cluster: c-000-000 -> Chọn bằng TF-IDF: ddos (Confidence: 0.52) | Similar: ['INC-2026-03-20', 'INC-2025-11-08', 'INC-2025-07-04']
Cluster: c-000-001 -> Chọn bằng TF-IDF: bad_deploy (Confidence: 0.48) | Similar: ['INC-2026-04-15', 'INC-2025-08-02', 'INC-2026-03-07']
Cluster: c-000-002 -> Chọn bằng TF-IDF: cache_cold_start (Confidence: 0.49) | Similar: ['INC-2026-05-25', 'INC-2025-12-01', 'INC-2026-01-29']
